In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, roc_curve, auc
from ISLP import load_data

In [ ]:
# Load the dataset
data = pd.read_csv("MBA.csv")  # Update with the correct file path

data['admission'] = data['admission'].replace({'Admit': 'Yes', 'Waitlist': '2', None: 'No'})
#data['admission'] = data['admission'].replace({'Admit': 1, 'Waitlist': '2', None: 0})

# Preprocess the data (drop missing values or handle them accordingly)
data = data.dropna()

# Drop rows where 'admission' column equals 2
data = data[data['admission'] != '2']

# Define features (X) and target (y)
from ISLP.models import (ModelSpec as MS, summarize, poly)
X = data.drop(columns=['admission', 'application_id'])
# Assuming 'X' is your DataFrame
X = pd.get_dummies(X, drop_first=True)
X = MS(X).fit_transform(X)
y = data['admission']

# Split the data into training and testing sets
# Set a random seed for reproducibility
np.random.seed(88)
train_size = int(len(X) / 3)
train_indices = np.random.choice(len(X), train_size, replace=False)
test_indices = np.setdiff1d(np.arange(len(X)), train_indices)

X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]

In [ ]:
### Logistic Regression Model
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# Predict probabilities on the test set
log_prob = log_reg.predict_proba(X_test)[:, 1]  # Get probabilities for the 'Yes' class
# Set prediction threshold at 0.5
log_pred = np.where(log_prob > 0.5, 'Yes', 'No')

In [ ]:
# Calculate the confusion matrix
log_cm = confusion_matrix(y_test, log_pred)

# Convert the confusion matrix into a DataFrame for better readability with labels
log_cm_df = pd.DataFrame(log_cm, index=['True: No', 'True: Yes'], columns=['Pred: No', 'Pred: Yes'])

# Print the confusion matrix with labels
print("Confusion Matrix:")
print(log_cm_df)


In [ ]:
# Extract values from the confusion matrix
TN, FP, FN, TP = log_cm.ravel()

In [ ]:
# Calculate False Positive Rate, False Negative Rate, True Positive Rate
false_positive_rate = FP / (FP + TN)  # FPR = FP / (FP + TN)
false_negative_rate = FN / (FN + TP)  # FNR = FN / (FN + TP)
true_positive_rate = TP / (TP + FN)   # TPR (Recall) = TP / (TP + FN)
true_negative_rate = TN / (TN + FP)   # TNR = TN / (TN + FP)

print(f"False Positive Rate (FPR): {false_positive_rate:.2f}")
print(f"False Negative Rate (FNR): {false_negative_rate:.2f}")
print(f"True Positive Rate (TPR): {true_positive_rate:.2f}")
print(f"True Negative Rate (TNR): {true_negative_rate:.2f}")

In [ ]:
# Calculate performance metrics
accuracy = accuracy_score(y_test, log_pred)
precision = precision_score(y_test, log_pred, pos_label='Yes')  # Adjust pos_label based on your dataset
recall = recall_score(y_test, log_pred, pos_label='Yes')
f1 = f1_score(y_test, log_pred, pos_label='Yes')
#Harmonic mean of precision and recall, balancing the two: f1=2*(precision*recall)/(precision+recall)
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall (TPR): {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, log_prob, pos_label='Yes')
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

